# Regressione delta_tas vs delta_cover, stagionale (adattata)

Analogo stagionale di `../01-cover_tas_regression_adapted.ipynb` (disegno non
circolare, vedi quel notebook per la spiegazione completa). Aggiunge il loop
sulle 4 stagioni (DJF/MAM/JJA/SON), oltre alle 10 combinazioni di lead year e
a {cvh, cvl}.

**IMPORTANTE**: eseguire prima la cella di verifica (`debug_years`) qui sotto e
controllare che gli anni di `cover DJF` e `tas DJF` corrispondano davvero allo
stesso inverno, prima di lanciare il calcolo completo. Vedi la docstring di
`cover_tas_seasons_lib.py` per il dettaglio del rischio (dicembre appartiene
all'anno solare precedente rispetto a gennaio-febbraio: un disallineamento di
un anno per DJF non darebbe errore, solo un risultato silenziosamente sbagliato).


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
variables = ['cvh', 'cvl']
SAVE_PATH = str(FIG_DIR / "01_adapted_seasons")  # sottocartella dedicata a questo notebook
os.makedirs(SAVE_PATH, exist_ok=True)


In [ ]:
sys.path.insert(0, os.getcwd())
from cover_tas_seasons_lib import run_one_adapted_season, debug_years, LEADS, SEASONS


In [ ]:
# --- VERIFICA PRIMA DI LANCIARE IL CALCOLO COMPLETO ---
# controlla che gli anni di cover e tas per DJF rappresentino lo stesso inverno
debug_years(exp_ctrl, exp_sens, 'cvh', 0, 4, 'DJF')
print()
debug_years(exp_ctrl, exp_sens, 'cvh', 0, 4, 'JJA')  # stagione senza ambiguita', per confronto


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, season, y1, y2, SAVE_PATH)
        for var in variables for season in SEASONS for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_adapted_season, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
